In [ ]:
"""
Fine-tune RoBERTa-large on n=50+101 synthetic training set.
Saves model checkpoint for prediction.
"""

import torch
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    RobertaTokenizer, RobertaForSequenceClassification,
    get_linear_schedule_with_warmup
)
import os

## Config matches your CV setup
MODEL_NAME = "roberta-large"
LR = 3e-5
EPOCHS = 4
BATCH_SIZE = 32
MAX_LEN = 75
SEED = 42
OUTPUT_DIR = "model_50_101"

## Set seeds
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


In [ ]:
## Load training data
df = pd.read_excel("../1. train_data/train_50_101.xlsx")
print(f"Training on {len(df)} examples")
print(f"  Positive: {(df['nostalgic']==1).sum()}")
print(f"  Negative: {(df['nostalgic']==0).sum()}")
n_pos = (df["nostalgic"]==1).sum()  # should be 151
n_neg = (df["nostalgic"]==0).sum()  # should be 449
class_weights = torch.tensor([1.0, n_neg / n_pos], dtype=torch.float).to(device)


In [ ]:
## Tokenizer
tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.long),
        }

train_ds = TextDataset(
    df["text"].tolist(),
    df["nostalgic"].astype(int).tolist(),
    tokenizer,
    MAX_LEN,
)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)


In [ ]:
## Model
model = RobertaForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(device)

## Optimizer / scheduler
total_steps = len(train_loader) * EPOCHS
optimizer = AdamW(model.parameters(), lr=LR)
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=0, num_training_steps=total_steps
)

## Train
model.train()
for epoch in range(EPOCHS):
    total_loss = 0
    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    avg = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{EPOCHS}  avg loss = {avg:.4f}")


In [ ]:
## Save
os.makedirs(OUTPUT_DIR, exist_ok=True)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}/")
